# Parte 2 - Denoising con COVID
Hecho por Samuel Patiño y Nicolás Peña - IA Generativa UAO

Aca usamos el dataset de rayos X de COVID. Lo bajamos directo en Kaggle con kagglehub, le metemos ruido gaussiano y entrenamos el mismo autoencoder del 01 pero para limpiar.

En Kaggle: Settings > Accelerator > GPU y Internet ON, si no el download falla.

In [ ]:
# revisamos GPU rapido
import tensorflow as tf
print(tf.__version__)
print(tf.config.list_physical_devices('GPU'))

In [ ]:
# librerias basicas
import os, numpy as np, matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Conv2DTranspose, BatchNormalization, Dropout, LeakyReLU, Dense
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import load_img, img_to_array
%matplotlib inline
np.random.seed(42)
tf.random.set_seed(42)

In [ ]:
# bajamos el dataset sin subir zip a mano
import kagglehub
base = kagglehub.dataset_download("pranavraikokte/covid19-image-dataset")
print("Path to dataset files:", base)
# miramos que carpetas trae
for root, dirs, files in os.walk(base):
    level = root.replace(base, "").count(os.sep)
    if level < 3:
        print(root, "dirs:", dirs[:5], "files:", len(files))
    if level > 3:
        break

In [ ]:
# contamos cuantas hay por clase para el informe
import pathlib
for split in ["train", "test"]:
    print(split)
    for clase in sorted(os.listdir(os.path.join(base, split))):
        n = len(os.listdir(os.path.join(base, split, clase)))
        print(" ", clase, n)

In [ ]:
# cargamos todo a 64x64 en rgb para reusar la arquitectura
# son grises pero las pedimos en rgb asi quedan de 64x64x3 como pide la parte 0
IMG = 64
def cargar(split):
    xs, ys, clases = [], [], sorted(os.listdir(os.path.join(base, split)))
    mapa = {c:i for i,c in enumerate(clases)}
    for c in clases:
        carpeta = os.path.join(base, split, c)
        for f in os.listdir(carpeta):
            p = os.path.join(carpeta, f)
            try:
                img = load_img(p, target_size=(IMG, IMG), color_mode="rgb")
                xs.append(img_to_array(img) / 255.0)
                ys.append(mapa[c])
            except:
                pass
    return np.array(xs, dtype="float32"), np.array(ys), clases
x_train, y_train, clases = cargar("train")
x_test, y_test, _ = cargar("test")
print(x_train.shape, x_test.shape, clases)
# mostramos 8 limpias
plt.figure(figsize=(12,3))
for i in range(8):
    ax = plt.subplot(1,8,i+1); plt.imshow(x_train[i]); plt.axis("off")
    ax.set_title(clases[y_train[i]], fontsize=7)
plt.show()

In [ ]:
# ruido gaussiano que pide el examen, tal cual el snippet del PDF
def add_noise(img, noise_factor=0.5):
    noisy_img = img + noise_factor * np.random.randn(*img.shape)
    noisy_img = np.clip(noisy_img, 0., 1.)
    return noisy_img
# lo aplicamos a todo
x_train_n = np.array([add_noise(im) for im in x_train])
x_test_n = np.array([add_noise(im) for im in x_test])
# vemos original vs ruidosa
plt.figure(figsize=(12,4))
for i in range(6):
    ax = plt.subplot(2,6,i+1); plt.imshow(x_train[i]); plt.axis("off")
    if i==0: ax.set_title("limpia")
    ax = plt.subplot(2,6,i+7); plt.imshow(x_train_n[i]); plt.axis("off")
    if i==0: ax.set_title("ruidosa")
plt.show()

## Autoencoder para limpiar
Es el mismo del 01 pero en 64x64. Baja 64 a 32 a 16 a 8 y sube al reves. Entrena con ruidosa de entrada y limpia de target.

In [ ]:
# encoder en 64
inputs = Input(shape=(64,64,3))
x = Conv2D(32,3,activation="relu",padding="same")(inputs)
x = BatchNormalization()(x)
x = MaxPooling2D()(x) # 32
x = Dropout(0.3)(x)
skip = Conv2D(32,3,padding="same")(x)
x = LeakyReLU()(skip)
x = BatchNormalization()(x)
x = MaxPooling2D()(x) # 16
x = Dropout(0.3)(x)
x = Conv2D(64,3,activation="relu",padding="same")(x)
x = BatchNormalization()(x)
x = MaxPooling2D()(x) # 8
flat = tf.keras.layers.Flatten()(x)
latent = Dense(256,activation="relu",name="latent")(flat)
encoder = Model(inputs, latent)
# decoder espejo
x = Dense(8*8*64,activation="relu")(latent)
x = tf.keras.layers.Reshape((8,8,64))(x)
x = Conv2DTranspose(64,3,activation="relu",strides=(2,2),padding="same")(x) # 16
x = BatchNormalization()(x)
x = Conv2DTranspose(32,3,activation="relu",strides=(2,2),padding="same")(x) # 32
x = BatchNormalization()(x)
x = Conv2DTranspose(32,3,padding="same")(x)
x = tf.keras.layers.Add()([x, skip])
x = LeakyReLU()(x)
x = BatchNormalization()(x)
decoded = Conv2DTranspose(3,3,activation="sigmoid",strides=(2,2),padding="same")(x) # 64
ae = Model(inputs, decoded)
ae.compile(optimizer=Adam(learning_rate=0.001), loss="mse")
ae.summary()

In [ ]:
# entrenamos con ruidosas de input y limpias de target
# como son poquitas usamos batch 32 y 30 epocas, en T4 son unos minutos
h = ae.fit(x_train_n, x_train, epochs=30, batch_size=32, shuffle=True, validation_data=(x_test_n, x_test))

In [ ]:
# curva
plt.figure()
plt.plot(h.history["loss"], label="train")
plt.plot(h.history["val_loss"], label="val")
plt.xlabel("epocas"); plt.ylabel("mse"); plt.legend(); plt.grid(alpha=0.3)
plt.title("Perdida denoising COVID")
plt.savefig("/kaggle/working/Loss_curve_part2.pdf", format="pdf")
plt.show()

In [ ]:
# tripleta original ruidosa denoised
pred = ae.predict(x_test_n[:10])
plt.figure(figsize=(15,6))
for i in range(10):
    ax = plt.subplot(3,10,i+1); plt.imshow(x_test[i]); plt.axis("off")
    if i==0: ax.set_title("original")
    ax = plt.subplot(3,10,i+11); plt.imshow(x_test_n[i]); plt.axis("off")
    if i==0: ax.set_title("ruidosa")
    ax = plt.subplot(3,10,i+21); plt.imshow(pred[i]); plt.axis("off")
    if i==0: ax.set_title("denoised")
plt.savefig("/kaggle/working/triplet_part2.pdf", format="pdf")
plt.show()
# este es el que va al informe, si queda ruido residual se dice tal cual

## Lo que vimos
Si el denoised se ve mas suave que la ruidosa y parecido a la original, va bien. Como son poquitas imagenes se sobreajusta facil, por eso usamos Dropout y batch chico. Si queda ruido residual se pone en la discusion y no se esconde.